In [ ]:
from pathlib import Path


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for path in (current, *current.parents):
        if (path / "data").exists():
            return path
    return current


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"

import json
import re
from pathlib import Path

import pandas as pd

DAY_DIR = DATA_ROOT / "data_gene" / "daily_data_gen"
ZH_DIR = DATA_ROOT / "data_preprocessing" / "process" / "industry" / "zhongxin_source"
OUT_DIR = DATA_ROOT / "data_preprocessing" / "process" / "industry" / "day_with_zhongxin"
CACHE_DIR = DATA_ROOT / "data_preprocessing" / "process" / "industry" / "cache"
ZH_ALL_CACHE = CACHE_DIR / "zhongxin_all.parquet"
CODE2TS_CACHE = CACHE_DIR / "code2ts_from_zhongxin.json"
START_DATE = "20150101"
END_DATE = "20251231"
SKIP_EXISTING_OUTPUT = True
FORCE_REBUILD_CACHE = False
PARQUET_ENGINE = "pyarrow"
PARQUET_COMPRESSION = "snappy"
DATE_RE = re.compile(r"(\d{8})")


def parse_day(path):
    match = DATE_RE.search(path.name)
    return int(match.group(1)) if match else None


def code6(value):
    text = str(value).strip()
    text = re.sub(r"\.0$", "", text)
    return text.zfill(6) if text.isdigit() and len(text) < 6 else text


def guess_ts_code(value):
    code = code6(value)
    if not code.isdigit() or len(code) != 6:
        return None
    if code.startswith("6"):
        return f"{code}.SH"
    if code.startswith(("0", "3")):
        return f"{code}.SZ"
    if code.startswith(("4", "8", "9")):
        return f"{code}.BJ"
    return None


def write_parquet(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    frame.to_parquet(temp_path, engine=PARQUET_ENGINE, compression=PARQUET_COMPRESSION, index=False)
    temp_path.replace(path)


def daily_files():
    files = []
    for path in sorted(DAY_DIR.glob("*.parquet")):
        day = parse_day(path)
        if day is not None and int(START_DATE) <= day <= int(END_DATE):
            files.append((day, path))
    return files


def build_code_map():
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    if CODE2TS_CACHE.exists():
        return json.loads(CODE2TS_CACHE.read_text(encoding="utf-8"))
    mapping = {}
    for path in sorted(ZH_DIR.glob("*.parquet")):
        ts_code = path.stem.upper()
        mapping[code6(ts_code.split(".")[0])] = ts_code
    CODE2TS_CACHE.write_text(json.dumps(mapping, indent=2), encoding="utf-8")
    return mapping


def load_industry_table():
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    if ZH_ALL_CACHE.exists() and not FORCE_REBUILD_CACHE:
        return pd.read_parquet(ZH_ALL_CACHE)
    frames = []
    for path in sorted(ZH_DIR.glob("*.parquet")):
        frame = pd.read_parquet(path)
        if frame.empty:
            continue
        if "ts_code" not in frame.columns:
            frame["ts_code"] = path.stem.upper()
        frames.append(frame)
    if not frames:
        raise RuntimeError(f"no industry files found: {ZH_DIR}")
    data = pd.concat(frames, ignore_index=True)
    data["ts_code"] = data["ts_code"].astype(str).str.upper()
    data["in_date"] = pd.to_datetime(data["in_date"].astype(str), errors="coerce")
    data["out_date"] = pd.to_datetime(data.get("out_date", pd.Series(pd.NA, index=data.index)).astype(str).replace({"": pd.NA, "nan": pd.NA}), errors="coerce")
    data = data.dropna(subset=["ts_code", "in_date"])
    keep = ["ts_code", "in_date", "out_date", "l1_name", "l2_name", "l3_name"]
    data = data[[col for col in keep if col in data.columns]].sort_values(["ts_code", "in_date"])
    write_parquet(data, ZH_ALL_CACHE)
    return data


def merge_day(day, path, industry, code_map):
    out_path = OUT_DIR / path.name
    if out_path.exists() and SKIP_EXISTING_OUTPUT:
        return "skip"
    frame = pd.read_parquet(path)
    code_col = "code" if "code" in frame.columns else "Code"
    date_col = "date" if "date" in frame.columns else "Date"
    frame["code6"] = frame[code_col].map(code6)
    frame["ts_code"] = frame["code6"].map(code_map)
    frame["ts_code"] = frame["ts_code"].fillna(frame["code6"].map(guess_ts_code))
    frame["trade_date"] = pd.to_datetime(frame[date_col].astype(str), errors="coerce")
    left = frame.reset_index(names="row_id").sort_values(["ts_code", "trade_date"])
    right = industry.sort_values(["ts_code", "in_date"])
    merged = pd.merge_asof(left, right, left_on="trade_date", right_on="in_date", by="ts_code", direction="backward")
    out_date = merged["out_date"].fillna(pd.Timestamp("2262-04-11"))
    valid = merged["trade_date"].lt(out_date)
    merged["ci_l1"] = merged["l1_name"].where(valid)
    merged["ci_l2"] = merged["l2_name"].where(valid)
    merged["ci_l3"] = merged["l3_name"].where(valid)
    merged = merged.sort_values("row_id")
    result = frame.drop(columns=["code6", "ts_code", "trade_date"], errors="ignore").copy()
    result["ci_l1"] = merged["ci_l1"].to_numpy()
    result["ci_l2"] = merged["ci_l2"].to_numpy()
    result["ci_l3"] = merged["ci_l3"].to_numpy()
    write_parquet(result, out_path)
    return "write"


def main():
    files = daily_files()
    code_map = build_code_map()
    industry = load_industry_table()
    counts = {"write": 0, "skip": 0}
    for day, path in files:
        status = merge_day(day, path, industry, code_map)
        counts[status] = counts.get(status, 0) + 1
        print(f"day={day} status={status}")
    print(f"done write={counts.get('write', 0)} skip={counts.get('skip', 0)} output={OUT_DIR}")


if __name__ == "__main__":
    main()